In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from retrieve.get_data import get_yield
from statsmodels.tsa.api import VAR

In [ ]:
# load yield
term = 1
df = get_yield(term)

df_prices = 100 / ((1 + df.copy()) ** term)
df = df.diff().dropna()

df['US'] = df['US'].shift(1)

In [ ]:
target = "AUS"      # we trade UK rates
p = 5              # VAR lag length (chosen earlier via AIC)
k_vol = 0.5        # only trade if forecast > 0.5 * normal weekly move
vol_lookback = 104 # 2-year rolling volatility estimate
tcost = 0.0        # optional transaction cost penalty
vol_target = 0.0   # optional risk scaling

idx = df.index
y_target = df[target].copy()

# typical weekly UK move size — used as noise filter
rolling_vol = y_target.rolling(vol_lookback).std()

# storage for model forecast and trading position
pred = pd.DataFrame(index=idx, dtype=float)
pos = pd.DataFrame(index=idx, dtype=float)
money = pd.DataFrame(index=idx, dtype=float)

In [ ]:
# 70/30 train/test split
n = len(df)
split_i = int(np.floor(0.70 * n))
split_date = idx[split_i]
start_i = max(split_i, p + 1, vol_lookback + 1)  # we need enough observations to estimate VAR, volatility

In [ ]:
portfolio = 100000.0 
pos = 0
Long = True

for i in range(start_i, len(idx)):
    end_date = idx[i]

    # expanding training window through end_date
    train = df.loc[:end_date].copy()

    # clean
    train = train.replace([np.inf, -np.inf], np.nan).dropna()

    # skip if not enough rows
    if len(train) <= p:
        pred.loc[end_date] = np.nan

    # skip if target missing
    if target not in train.columns:
        pred.loc[end_date] = np.nan
    
    model = VAR(train)
    res = model.fit(p)

    # 1-step-ahead forecast
    fcast = res.forecast(train.values[-p:], steps=1)[0]

    # if fcast > rolling_vol.loc[end_date] * k_vol:
    #     #buy the bond using the entire cash position.
    #     continue
    # else:  
    #     #sell the bond and move to cash.
    #     continue


    # grab target forecast
    # j = res.names.index(target)
    # pred.loc[end_date] = fcast[j]
    # print(cash)

In [ ]:
bond_price = df_prices[target]

portfolio = 100000.0 
pos = 0
Long = True

starting_portfolio = 100000.0

portfolio.loc[idx[start_i]] = starting_portfolio

for i in range(start_i, len(idx) - 1):
    end_date = idx[i]
    next_date = idx[i + 1]

    # ---- VAR stuff (same as before) ----
    train = df.loc[:end_date].copy()
    train = train.replace([np.inf, -np.inf], np.nan).dropna()

    n_vars = train.shape[1]
    min_rows = max(30, p + 5, n_vars * p + 5)
    if len(train) < min_rows:
        pos.loc[end_date] = 0.0
        portfolio.loc[next_date] = portfolio.loc[end_date]
        continue

    constant_cols = train.columns[train.nunique() <= 1]
    if len(constant_cols) > 0:
        train = train.drop(columns=constant_cols)

    if target not in train.columns:
        pos.loc[end_date] = 0.0
        portfolio.loc[next_date] = portfolio.loc[end_date]
        continue

    try:
        res = VAR(train).fit(p)
        fcast = res.forecast(y=train.values[-p:], steps=1)[0]
        j = res.names.index(target)
        pred.loc[end_date] = fcast[j]
    except:
        pos.loc[end_date] = 0.0
        portfolio.loc[next_date] = portfolio.loc[end_date]
        continue

    # ---- position logic ----
    vol = rolling_vol.loc[end_date]
    if pd.isna(vol) or vol == 0:
        pos.loc[end_date] = 0.0
    elif abs(pred.loc[end_date]) < k_vol * vol:
        pos.loc[end_date] = 0.0
    else:
        # yield up -> short bond
        pos.loc[end_date] = -np.sign(pred.loc[end_date]) * position_size

    # ---- price-based PnL ----
    if next_date in bond_price.index:
        price_today = bond_price.loc[end_date]
        price_next = bond_price.loc[next_date]

        pnl = pos.loc[end_date] * (price_next - price_today)

        portfolio.loc[next_date] = portfolio.loc[end_date] + pnl
    else:
        portfolio.loc[next_date] = portfolio.loc[end_date]

# fill forward
portfolio = portfolio.ffill()

ending_portfolio = portfolio.dropna().iloc[-1]
total_return = ending_portfolio / starting_portfolio - 1

print(f"Starting portfolio: ${starting_portfolio:,.2f}")
print(f"Ending portfolio:   ${ending_portfolio:,.2f}")
print(f"Total return:       {total_return:.2%}")

In [ ]:
# PnL approximation
# price return ≈ -Δyield
# so long duration profits when yields fall
pnl = pos.shift(1) * (-y_target)

# turnover cost penalty
turnover = (pos - pos.shift(1)).abs().fillna(0.0)
pnl = pnl - tcost * turnover

# evaluate out-of-sample only
test_mask = idx >= split_date
pnl_oos = pnl.loc[test_mask].dropna()
cum = pnl_oos.cumsum()

In [ ]:
metrics = pd.concat([pred, pos, money], axis=1, keys=['pred','pos','money']).sort_index()

In [ ]:
# performance
mean_w = pnl_oos.mean()
std_w = pnl_oos.std(ddof=0)

sharpe = (mean_w / std_w) * np.sqrt(252) if std_w > 0 else np.nan
hit_rate = (pnl_oos > 0).sum() / (pnl_oos != 0).sum()
max_dd = (cum - cum.cummax()).min()

trades = int((turnover.loc[pnl_oos.index] > 0).sum())
pct_in = (pos.loc[pnl_oos.index] != 0).mean()

print(f"Target:              {target}")
print(f"Test window:         {pnl_oos.index.min().date()} -> {pnl_oos.index.max().date()}")
print(f"Ann. Sharpe:         {sharpe:.2f}")
print(f"Mean weekly PnL:     {mean_w:.5f}")
print(f"Cumulative PnL:      {cum.iloc[-1]:.3f}")
print(f"Max drawdown:        {max_dd:.3f}")
print(f"Hit rate:            {hit_rate:.1%}")
print(f"Trades:              {trades}")
print(f"% weeks in market:   {pct_in:.1%}")

In [ ]:
# cumulative PnL plot
plt.figure(figsize=(12, 5))
plt.plot(cum.index, cum.values)
plt.axhline(0, color="gray", linestyle="--", linewidth=0.5)
plt.title(f"VAR Spillover Strategy — {target} (OOS)")
plt.xlabel("Date")
plt.ylabel("Cumulative PnL (yield units)")
plt.tight_layout()
plt.show()